In [2]:
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor # Import RandomForestClassifier
from xgboost import XGBClassifier,XGBRegressor # Import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler # For scaling numerical features

In [3]:
final_df = pd.read_csv("Final_Reviews.csv")
final_df

,review_text,review_length,feedback,play_hours,date_posted,day_of_week,month,processed_review_text,vader_sentiment_scores
0,10 out of 10 game but MJ is the main reason wh...,67,1,44.8,2025-07-06,Sunday,July,game mj main reason havent game year yet,0.0000
1,best game ever,12,1,16.1,2025-07-06,Sunday,July,best game ever,0.6369
2,"this game is very good, is an anventure game i...",43,1,5.0,2025-07-06,Sunday,July,game good anventure game like,0.6597
3,not well optimized for now,22,0,2.5,2025-07-06,Sunday,July,not well optimized,-0.5096
4,cause,5,1,7.2,2025-07-06,Sunday,July,cause,0.0000
...,...,...,...,...,...,...,...,...,...
1986,"The story in this game is disappointing, Krave...",847,0,27.7,2025-04-09,Wednesday,April,story game disappointing kraven absolutely was...,0.9424
1987,My favourite spider man game by far ^^\n\nI lo...,561,1,41.0,2025-04-09,Wednesday,April,favourite spider man game far love improvement...,0.9771
1988,good game. just wished it wasnt so heavy on th...,57,1,34.1,2025-04-09,Wednesday,April,good game wished wasnt heavy lgbtqwoke,0.4404
1989,Not as good as the first one but a great game ...,140,1,26.0,2025-04-09,Wednesday,April,not good first one great game great gameplay a...,0.8997


In [4]:
X_feedback = final_df[['vader_sentiment_scores', 'review_length']]
y_feedback = final_df['feedback']

scaler_feedback = StandardScaler()
X_feedback_scaled = scaler_feedback.fit_transform(X_feedback)

X_train_feedback, X_test_feedback, y_train_feedback, y_test_feedback = train_test_split(
    X_feedback_scaled, y_feedback, test_size=0.2, random_state=42, stratify=y_feedback
)

# Calculating the scale_pos_weight for XGBoost to handle imbalance
# count_neg_class / count_pos_class
# In our case, scale_pos_weight = count(Not Recommended) / count(Recommended)
neg_count = y_train_feedback.value_counts()[0] 
pos_count = y_train_feedback.value_counts()[1]
scale_pos_weight_val = neg_count / pos_count
print(f"Calculated scale_pos_weight for XGBoost: {scale_pos_weight_val:.2f}")



models = {
    "Logistic Regression": LogisticRegression(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss',
                             scale_pos_weight=scale_pos_weight_val) # For imbalance
}

trained_feedback_models = {}

for name, model in models.items():
    print(f"\n--- Training {name} Model ---")
    model.fit(X_train_feedback, y_train_feedback)
    y_pred = model.predict(X_test_feedback)

    accuracy = accuracy_score(y_test_feedback, y_pred)
    report = classification_report(y_test_feedback, y_pred)

    print(f"Feedback Prediction Model ({name}) Accuracy: {accuracy:.2f}")
    print(f"Feedback Prediction Model ({name}) Classification Report:\n", report)

    trained_feedback_models[name] = model


def predict_feedback(new_review_text: str, model_name: str, models_dict, scaler, sid_analyzer):
    """
    Predicts feedback (Recommended/Not Recommended) for a new review text using a specified model.

    Args:
        new_review_text (str): The text of the new review.
        model_name (str): The name of the model to use (e.g., "Logistic Regression", "Random Forest", "XGBoost").
        models_dict (dict): Dictionary containing trained models.
        scaler: The StandardScaler fitted on the training data.
        sid_analyzer: The VADER SentimentIntensityAnalyzer.

    Returns:
        str: 'Recommended' or 'Not Recommended'.
        float: The VADER compound score for the review.
        int: The length of the review text.
    """
    if model_name not in models_dict:
        raise ValueError(f"Model '{model_name}' not found in trained models.")

    model = models_dict[model_name]

    #VADER score
    vader_score = sid_analyzer.polarity_scores(new_review_text)['compound']
    #review length
    review_len = len(new_review_text)

    #DataFrame for the new input, matching training features
    new_input_df = pd.DataFrame([[vader_score, review_len]], columns=['vader_sentiment_scores', 'review_length'])

    # Scale the new input using the *same* scaler used for training
    new_input_scaled = scaler.transform(new_input_df)

    # Predict [0] to get the single prediction
    prediction = model.predict(new_input_scaled)[0]

    return 'Recommended' if prediction == 1 else 'Not Recommended', vader_score, review_len


Calculated scale_pos_weight for XGBoost: 0.22

--- Training Logistic Regression Model ---
Feedback Prediction Model (Logistic Regression) Accuracy: 0.70
Feedback Prediction Model (Logistic Regression) Classification Report:
               precision    recall  f1-score   support

           0       0.35      0.71      0.47        73
           1       0.92      0.70      0.79       326

    accuracy                           0.70       399
   macro avg       0.63      0.71      0.63       399
weighted avg       0.81      0.70      0.73       399


--- Training Random Forest Model ---
Feedback Prediction Model (Random Forest) Accuracy: 0.84
Feedback Prediction Model (Random Forest) Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.52      0.55        73
           1       0.89      0.91      0.90       326

    accuracy                           0.84       399
   macro avg       0.74      0.72      0.73       399
weighted avg   

/Users/mac/Desktop/sentimentAnalysis/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [21:58:01] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [5]:
sia = SentimentIntensityAnalyzer()

test_review_sample = "This game is a masterpiece! I've been waiting for this for years and it exceeded all expectations. Highly recommended to everyone."

print(f"\nReview to test: '{test_review_sample}'")

for name in trained_feedback_models.keys():
    pred, vader_score, review_len = predict_feedback(test_review_sample, name, trained_feedback_models, scaler_feedback, sia)
    print(f"  Using {name}: Predicted Feedback: {pred}")


Review to test: 'This game is a masterpiece! I've been waiting for this for years and it exceeded all expectations. Highly recommended to everyone.'
  Using Logistic Regression: Predicted Feedback: Recommended
  Using Random Forest: Predicted Feedback: Recommended
  Using XGBoost: Predicted Feedback: Recommended


In [ ]:
from sklearn.preprocessing import StandardScaler

X_feedback = final_df[['vader_sentiment_scores', 'review_length']]
y_feedback = final_df['feedback']

# Scale numerical features
scaler_feedback = StandardScaler()
X_feedback_scaled = scaler_feedback.fit_transform(X_feedback)

# Split data into training and testing sets
X_train_feedback, X_test_feedback, y_train_feedback, y_test_feedback = train_test_split(
    X_feedback_scaled, y_feedback, test_size=0.2, random_state=42, stratify=y_feedback
)

neg_count = y_train_feedback.value_counts()[0] # Count of 'Not Recommended'
pos_count = y_train_feedback.value_counts()[1] # Count of 'Recommended'
scale_pos_weight_val = neg_count / pos_count
print(f"Calculated scale_pos_weight for XGBoost: {scale_pos_weight_val:.2f}")

model_feedback = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss',
                             scale_pos_weight=scale_pos_weight_val) # For imbalance

model_feedback.fit(X_train_feedback, y_train_feedback)

# Make predictions on the test set
y_pred_feedback = model_feedback.predict(X_test_feedback)

# Evaluate the model
accuracy_feedback = accuracy_score(y_test_feedback, y_pred_feedback)
report_feedback = classification_report(y_test_feedback, y_pred_feedback)

print(f"Feedback Prediction Model Accuracy: {accuracy_feedback:.2f}")
print("Feedback Prediction Model Classification Report:\n", report_feedback)

# --- Prediction Function for Feedback ---
def predict_feedback(new_review_text: str, model, scaler, sia_analyzer):
    """
    Predicts feedback (Recommended/Not Recommended) for a new review text.

    Args:
        new_review_text (str): The text of the new review.
        model: The trained XGBoost model.
        scaler: The StandardScaler fitted on the training data.
        sid_analyzer: The VADER SentimentIntensityAnalyzer.

    Returns:
        str: 'Recommended' or 'Not Recommended'.
        float: The VADER compound score for the review.
        int: The length of the review text.
    """
    # Calculate VADER score
    vader_score = sia_analyzer.polarity_scores(new_review_text)['compound']
    # Calculate review length
    review_len = len(new_review_text)

    # Create a DataFrame for the new input, matching training features
    new_input_df = pd.DataFrame([[vader_score, review_len]], columns=['vader_sentiment_scores', 'review_length'])

    # Scale the new input using the *same* scaler used for training
    new_input_scaled = scaler.transform(new_input_df)

    # Predict
    prediction = model.predict(new_input_scaled)[0] # [0] to get the single prediction
    
    return 'Recommended' if prediction == 1 else 'Not Recommended', vader_score, review_len


Calculated scale_pos_weight for XGBoost: 0.22
Feedback Prediction Model Accuracy: 0.83
Feedback Prediction Model Classification Report:
               precision    recall  f1-score   support

           0       0.52      0.73      0.61        73
           1       0.93      0.85      0.89       326

    accuracy                           0.83       399
   macro avg       0.73      0.79      0.75       399
weighted avg       0.86      0.83      0.84       399



/Users/mac/Desktop/sentimentAnalysis/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [14:24:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [84]:
sia = SentimentIntensityAnalyzer()

test_review_1 = "This game is a masterpiece! I've been waiting for this for years and it exceeded all expectations. Highly recommended to everyone."
pred_feedback_1, vader_score_1, review_len_1 = predict_feedback(test_review_1, model_feedback, scaler_feedback, sia)
print(f"\nReview: '{test_review_1}'")
print(f"  VADER Score: {vader_score_1:.2f}, Length: {review_len_1}")
print(f"  Predicted Feedback: {pred_feedback_1}")


Review: 'This game is a masterpiece! I've been waiting for this for years and it exceeded all expectations. Highly recommended to everyone.'
  VADER Score: 0.76, Length: 130
  Predicted Feedback: Recommended
